# EUR_V1 — seizures found against the share of the recording reviewed (Track F, milestone M2, item V1, chat F2-02)

**Design:** `results/m2_v1/DESIGN_V1.md` (approved 23/11/2026 13:39). Inference only, the four committed V5 checkpoints:
V5's own label-free point (= EUR_ABL's V0 rows, gate G-v5own), a budget sweep 0.1–20 % of windows (detector and change
point stages; the committed curve rows reproduced, gate G-curve), a chance reference (scores circularly shifted within
each file, 10 shifts), the top-N events per hour of EEG (V5's own events; peak-centred 120-s clips), and the seizures
caught or missed at every budget. **Runs a SMOKE first** (chb01, chb03, chb24: every gate, measured) and STOPs before
the full run if anything differs from the V5 record. **Accelerator:** None (CPU). **Internet:** ON (pip, git clone).
**Inputs (Add Input):** `nhn2mm/v9-x3-fold1`, `nhannguyen26/v9-x3-fold3`, `norncreades/thesis-r2-external-features`.
**Output:** `/kaggle/working/eureka/v1`
(→ `results/m2_v1/run`). **Runtime:** ≈ 1–2 h (computed; the smoke writes `projection.json`).
Code: `github.com/Nhanvas/Thesis` ref `main`; every module's sha256 is asserted below.
**Nothing printed here is a research result.** Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "EUR_V1", "main"
TEST = bool(os.environ.get("EUR_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("EUR_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("EUR_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("EUR_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["EUR_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/eur_v1.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"eur_v1.py": "88161cb3bdfe2503", "eur_abl.py": "0fd303e9b2f3671f", "r2_r1.py": "76cbe51ff0180d4a", "r2_core.py": "902d15ad4feb3e8a", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "p3c_freeze.py": "dabe4139f6005fd2", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e22_test_oneshot.py": "669350bbfaaf0f4e", "e23_test_look7.py": "88587668baf12a64", "x1_external.py": "9962e3e7fe4ddc35", "p2_dump_seeds23.py": "0bc887542989310f", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "szcore_eval.py": "9cad5a6c9322c328"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import eur_v1 as M
assert M.__file__ == f"{REPO}/src/eur_v1.py", M.__file__
print("CPU cores:", os.cpu_count(), "| eur_v1", M.CODE_SHA, "| grid", M.GRID)

In [ ]:
# RUN: G-self-v1, G-data, SMOKE (R1.0 rows, G-v5own, G-curve, projection), then the full run
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("EUR_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, projection, outputs, measured runtime
OUT = os.path.join(WORK, "eureka", "v1")
for f in ("gates.json", "smoke/gates.json", "projection.json", "k.csv", "run_meta.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")